## Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 'processing-2.processing.data.production.internal',
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/geo-consistency'
print(local_dataset)

/Users/E2074/analytics/customer/segmentation/geo-consistency
/Users/E2074/local-datasets/customer/geo-consistency


## Datasets & Parameter

In [5]:
city = 'Bangalore'
date_str = '20250316'
date_obj = datetime.strptime(date_str, "%Y%m%d")

start_date = date_obj - timedelta(days=90)
end_date = date_obj

date_list = [(start_date + timedelta(days=i)).strftime("%Y%m%d") for i in range(91)]

print("start_date =", start_date.strftime("%Y%m%d"))
print("end_date =", end_date.strftime("%Y%m%d"))
print("date_range =", len(date_list))
print("list_date =", date_list)

start_date = 20241216
end_date = 20250316
date_range = 91
list_date = ['20241216', '20241217', '20241218', '20241219', '20241220', '20241221', '20241222', '20241223', '20241224', '20241225', '20241226', '20241227', '20241228', '20241229', '20241230', '20241231', '20250101', '20250102', '20250103', '20250104', '20250105', '20250106', '20250107', '20250108', '20250109', '20250110', '20250111', '20250112', '20250113', '20250114', '20250115', '20250116', '20250117', '20250118', '20250119', '20250120', '20250121', '20250122', '20250123', '20250124', '20250125', '20250126', '20250127', '20250128', '20250129', '20250130', '20250131', '20250201', '20250202', '20250203', '20250204', '20250205', '20250206', '20250207', '20250208', '20250209', '20250210', '20250211', '20250212', '20250213', '20250214', '20250215', '20250216', '20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223', '20250224', '20250225', '20250226', '20250227', '20250228', '20250301', '20250302', '2025

In [8]:
customer_base_query = f"""

    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
"""

df_base = pd.read_sql(customer_base_query, connection)

In [9]:
df_base.head(5)

,customer_id,segment
0,64e73688be1489cb87cbe91e,3-MONTHLY
1,6715f13569ec30689d3eb9ae,4-OTHER
2,641c3a92fb7d083c9fcd91cc,5-INACTIVE
3,6289fa23fb23d03fefe62319,3-MONTHLY
4,62f660e74d8a827e787fa596,3-MONTHLY


In [8]:
def get_data():
    for date in date_list:
    
        query_start_time = time.time()
        base_query = f"""
        
        select
            yyyymmdd,
            customer_id,
            service_obj_service_name service_name,
            order_id,
            order_status,
            pickup_cluster,
            drop_cluster,
            pickup_location_hex_8,
            drop_location_hex_8,
            pickup_location_hex_10,
            drop_location_hex_10,
            channel_host
        from 
            orders.order_logs_snapshot
        where 
            yyyymmdd = '{date}'
            and city_name = '{city}'
            and service_obj_service_name in ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'CabPremium', 'Cab SUV', 'Auto Pet', 'Auto C2C', 'Auto NCR','Auto Pool')
        """
        
        df_base_query = pd.read_sql(base_query, connection)
        query_end_time = time.time()
        
        extract_start_time = time.time()
        df_base_query.to_parquet(local_dataset + '/ban_gc_data_dump_{}.parquet'.format(date), index=False)
        extract_end_time = time.time()
        
        query_execution_time = query_end_time - query_start_time
        local_extract_time = extract_end_time - extract_start_time
        
        print(f"Query execution time: {query_execution_time:.6f} seconds & local extract time: {local_extract_time:.6f} seconds ")


# get_data()

In [6]:
def get_customer():
    
    parquet_files = glob.glob(os.path.join(local_dataset + "/*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)    
    df = df[df['order_status'] == 'dropped']
    return df[['yyyymmdd', 'customer_id', 'order_id', 
               # 'service_name', 'order_status',
               # 'pickup_cluster', 'drop_cluster', 'pickup_location_hex_12', 'drop_location_hex_12', 'channel_host',
               'pickup_tag', 'drop_tag']]

df_data = get_customer()

In [7]:
print(df_data.shape)
df_data.head(5)

(39389424, 5)


,yyyymmdd,customer_id,order_id,pickup_tag,drop_tag
1,20241216,616592c34c6ba133acd38677,67603c487f44a8713abc1c52,UNKNOWN,residential
2,20241216,65af7b3f1bf11283e3706d61,67601a5a10635f5fd6c1e8fb,residential,residential
4,20241216,5fe0922a4319a7a116b32751,675fc3ef1427162f4a691901,residential,leisure
7,20241216,66d5c990637b6ef7000fb97a,675fd247446da7409902665d,UNKNOWN,residential
8,20241216,67603be04ca2d022f19b4c39,67603c46347bdd18069041d9,UNKNOWN,transit_station


In [ ]:
# df_data\
# .groupby(['channel_host'])\
# .agg(customers = ('customer_id', 'nunique'), orders = ('order_id', 'nunique'))\
# .reset_index()

In [ ]:
# Excluding Tummoc channel_host 
# df_data = df_data[df_data['channel_host'].isin(['android','ios'])]

In [ ]:
# df_data\
# .groupby(['channel_host'])\
# .agg(customers = ('customer_id', 'nunique'), orders = ('order_id', 'nunique'))\
# .reset_index()

### Analysis

In [10]:
df_data['pickup_tag'].unique()

array(['UNKNOWN', 'residential', 'leisure', 'govt_amenity', None,
       'place_of_worship', 'hotels', 'office', 'household_needs',
       'banking', 'health_and_personal', 'transit_station', 'educational',
       'food', 'commercial'], dtype=object)

In [11]:
# df_data[(df_data['pickup_tag'] == 'office') | (df_data['drop_tag'] == 'office')]
df_data['office_tag'] = np.where((df_data['pickup_tag'] == 'office') | (df_data['drop_tag'] == 'office'), 1, None)
df_data['transit_tag'] = np.where((df_data['pickup_tag'] == 'transit_station') | (df_data['drop_tag'] == 'transit_station'), 1, None)

In [12]:
df_data.head(5)

,yyyymmdd,customer_id,order_id,pickup_tag,drop_tag,office_tag,transit_tag
1,20241216,616592c34c6ba133acd38677,67603c487f44a8713abc1c52,UNKNOWN,residential,None,None
2,20241216,65af7b3f1bf11283e3706d61,67601a5a10635f5fd6c1e8fb,residential,residential,None,None
4,20241216,5fe0922a4319a7a116b32751,675fc3ef1427162f4a691901,residential,leisure,None,None
7,20241216,66d5c990637b6ef7000fb97a,675fd247446da7409902665d,UNKNOWN,residential,None,None
8,20241216,67603be04ca2d022f19b4c39,67603c46347bdd18069041d9,UNKNOWN,transit_station,None,1


In [14]:
df_customer_office_bin = df_data\
                            .groupby(['customer_id'])\
                            .agg(total_net_orders = ('order_id', 'nunique'),
                                 office_orders = ('office_tag', 'sum'),
                                 transit_orders = ('transit_tag', 'sum'),
                                )\
                            .reset_index()

df_customer_office_bin

,customer_id,total_net_orders,office_orders,transit_orders
0,5737c6abddbec2203f733155,2,0,0
1,5737c6adddbec2203f73316a,9,0,1
2,5737c6adddbec2203f73316d,4,0,0
3,5737c6aeddbec2203f733176,24,1,0
4,5737c6b1ddbec2203f73318b,8,0,0
...,...,...,...,...
4310986,67d716dc810ce7da981873e7,1,1,0
4310987,67d716e4aebcc424855fa250,1,0,0
4310988,67d717937c212f7fecd46db6,1,1,0
4310989,67d7179559d5bfbb2ba40d74,1,1,0


In [15]:
df_customer_office_bin['office_orders'].describe()

count     4310991
unique        162
top             0
freq      2813604
Name: office_orders, dtype: int64

In [16]:
df_customer_office_bin['office_orders'].quantile([.25, .5, .75, 0.80, 0.85, 0.90])

0.25    0.0
0.50    0.0
0.75    1.0
0.80    1.0
0.85    2.0
0.90    3.0
Name: office_orders, dtype: object

In [17]:
df_customer_office_bin['transit_orders'].quantile([.25, .5, .75, 0.80, 0.85, 0.90])

0.25    0.0
0.50    0.0
0.75    1.0
0.80    1.0
0.85    1.0
0.90    2.0
Name: transit_orders, dtype: object

In [18]:
df_base.head(1)

,customer_id,segment
0,64e73688be1489cb87cbe91e,3-MONTHLY


In [19]:
df_export = pd.merge(df_base, df_customer_office_bin, how = 'left', on = 'customer_id')

In [20]:
df_export

,customer_id,segment,total_net_orders,office_orders,transit_orders
0,64e73688be1489cb87cbe91e,3-MONTHLY,8.0,5,0
1,6715f13569ec30689d3eb9ae,4-OTHER,1.0,0,0
2,641c3a92fb7d083c9fcd91cc,5-INACTIVE,NaN,NaN,NaN
3,6289fa23fb23d03fefe62319,3-MONTHLY,6.0,1,0
4,62f660e74d8a827e787fa596,3-MONTHLY,24.0,12,5
...,...,...,...,...,...
4929720,6579636cb65eb58d558f2966,4-OTHER,2.0,0,0
4929721,5ad22cae5008c43e3fca0118,3-MONTHLY,7.0,0,0
4929722,63cbda01d9b6ef4e1c55b55c,2-WEEKLY,35.0,29,0
4929723,673eebdc07b2b44b6deb2fe0,5-INACTIVE,NaN,NaN,NaN


In [21]:
df_base.shape, df_export.shape

((4929725, 2), (4929725, 5))

In [22]:
df_export.to_csv('office_transit_use_case_bins.csv', header=False, index=False)

In [23]:
df_export[df_export['customer_id'] == '67accc84be77c247c7d10003']

,customer_id,segment,total_net_orders,office_orders,transit_orders
3213299,67accc84be77c247c7d10003,4-OTHER,2.0,0,1
